In [2]:
!requirements.txt

zsh:1: command not found: requirements.txt


!requirements.txt


In [3]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass(" Enter GROQ_API_KEY: ")
if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass(" Enter TAVILY_API_KEY: ")
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass(" Enter PINECONE_API_KEY: ")

print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))

GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


In [4]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,) ,
    requests_kwargs={
        "headers": {
            "User-Agent":  "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)

raw_docs = loader.load()
print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])


/var/folders/6p/g8mlnqts4xd7zt96dc368w7h0000gn/T/ipykernel_2639/2794512715.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional AP

# Split Document into chunks


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks=splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])


Total chunks: 28

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional A


# Create Free Local Embedding

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print("Embedding dimensions:", len(sample_vector))

/Users/kailashmac/Documents/Agentic RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 20:18:59.534000 2639 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11021.87it/s]


Embedding dimensions: 384


In [7]:
sample_vector

[-0.06795874983072281,
 0.04574914276599884,
 -0.03998815268278122,
 -0.02448643185198307,
 -0.015574621967971325,
 0.011997463181614876,
 0.07909311354160309,
 0.056285448372364044,
 -0.026718800887465477,
 0.03930019959807396,
 0.04286826029419899,
 0.017323747277259827,
 0.023811105638742447,
 0.011839798651635647,
 -0.0260944627225399,
 0.05797801911830902,
 0.08081161975860596,
 0.09711682051420212,
 -0.03452671691775322,
 0.04521510750055313,
 -0.004475767258554697,
 0.044358737766742706,
 -0.012346627190709114,
 -0.03598394617438316,
 -0.046190619468688965,
 0.03801548480987549,
 -0.024391252547502518,
 0.008649869821965694,
 0.01989160105586052,
 -0.026702769100666046,
 0.029451265931129456,
 0.10643848031759262,
 0.00439863558858633,
 -0.04615795984864235,
 0.048952534794807434,
 0.017073769122362137,
 -0.07043419033288956,
 0.032346904277801514,
 0.07115577161312103,
 0.06753639876842499,
 0.04229779541492462,
 -0.023420283570885658,
 -0.02044256404042244,
 -0.059116575866937

# Create Pinecone Vector Database


In [8]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create the index only if it does not already exist.
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

''' The spec parameter specifies the infrastructure configuration for the index.

Here, you are choosing Pinecone's serverless deployment option.

Serverless means you do not need to manage your own dedicated vector-database servers. Pinecone manages the underlying infrastructure. '''


''' 
Traditional database                    Pinecone vector database

Database                                Pinecone service/project

Table or collection (rough analogy)     Index

Grouping or partition                   Namespace

Row or record (rough analogy)           Vector with an ID and optional metadata

Query                                   Vector similarity search

TABLE --> NAMESPACE

INDEX --> DATABASE
'''

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,       # all-MiniLM-L6-v2 embedding dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Wait until Pinecone reports the new index as ready.
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

print("Pinecone index ready:", INDEX_NAME)

# Upload the document chunks and create the LangChain vector store.
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

# k =4 means it will retrieve us 4 relevant documents after every similarity search

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Pinecone vector database and retriever are ready.")

Pinecone index ready: industry-agentic-rag-kb
Pinecone vector database and retriever are ready.


In [9]:
# Load Existing Index

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Load existing Pinecone index
index = pc.Index(INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Existing Pinecone index loaded:", INDEX_NAME)

Existing Pinecone index loaded: industry-agentic-rag-kb


# Test Private KB Retrieval

In [10]:
test_question="What happens if the retrieved documents are not relevant in Agentic RAG?"

kb_docs=retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):
    print(f"\n--- KB RESULT {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])


--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL as

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: 

# Initialise GROQ LLM

In [11]:
from langchain_groq import ChatGroq

# Replace this model if your Groq dashboard shows a different available model

llm = ChatGroq(
    model= "openai/gpt-oss-20b",
    temperature= 0,
)

print(llm.invoke("Explain Agentic RAG in one sentence."))



content='Agentic RAG is a retrieval‑augmented generation framework in which an autonomous agent actively selects, prioritizes, and incorporates relevant external documents to guide and improve its generated responses.' additional_kwargs={'reasoning_content': 'We need to explain Agentic RAG in one sentence. Agentic Retrieval-Augmented Generation? Agentic RAG refers to a system where the agent actively selects relevant documents to augment generation. So one sentence: "Agentic RAG is a retrieval-augmented generation approach where an autonomous agent proactively selects, prioritizes, and incorporates external knowledge sources to guide and improve its responses." That should be fine.'} response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 80, 'total_tokens': 208, 'completion_time': 0.146210931, 'completion_tokens_details': {'reasoning_tokens': 83}, 'prompt_time': 0.007282514, 'prompt_tokens_details': None, 'queue_time': 0.351319456, 'total_time': 0.153493445}, 'mo

In [12]:
print(llm.invoke("Explain Agentic RAG in one sentence.").content)

Agentic RAG is a retrieval‑augmented generation framework in which an autonomous agent actively selects, prioritizes, and incorporates relevant external documents to guide and improve its generated responses.


# Initialise Tavilty Websearch

In [13]:
from langchain_tavily import TavilySearch

''' max_results=5 means it will give 5 maximum relevant web evidance, from 5 different website it will give us the relevant info'''

websearch=TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=False,
)

web_search = websearch
print("Tavily Search tool ready")

Tavily Search tool ready


In [14]:
# All the utility funcations are ready now lets start with actual architecture implementation

#  WE need to define the pydantic structure and the state 

# To implelemnt Langgraph workflow we need a state (Agent state)

In [15]:
from typing import List,Literal,Optional
from typing_extensions import TypedDict
from pydantic import BaseModel,Field
from langchain_core.documents import Document


# Creating pydantic model validator especially for routing decision - it allows us to take a decision either a kb or direct

class RouteDecision(BaseModel):
    route: Literal["kb", "direct"] = Field(
        description="Use kb for questions needing Agentic RAG docs; direct for simple chat/greetings"
    )


# Creating pydantic model validator especially for routing decision - it checks the generated evidence is weak or good

class EvidenceGrade(BaseModel):
    grade: Literal["good", "weak"] = Field(
        description="good meand evidence can answer the question;weak means not enough evidence"
    )




# Rewrite query is basically the current query here
# from typing we are using TypedDict
class AgentState(TypedDict):
    question: str
    current_query: str
    kb_docs: List[Document]
    web_results: str
    kb_grade: str
    web_grade: str
    answer: str
    source_used: str
    retry_count: int 


<!-- Node 1 Route the Question -->

In [16]:

router_llm = llm.with_structured_output(RouteDecision, method="json_mode")

def route_question(state: AgentState):
    question = state["question"]

    decision = router_llm.invoke(f'''
You are a router for an Agentic RAG assistant.

Route to "kb" if the user asks about:
- Agentic RAG
- LangGraph Agentic RAG workflow
- retrieval grading
- query rewriting
- RAG architecture
- retriever tools
- web fallback in RAG

Route to "direct" only for greetings, thanks, or very simple conversation.

Question:
{question}

Return your response as valid JSON.
Example:
{{"route": "kb"}}
''')

    print("[Router]", decision.route)

    return {
        "current_query": question,
        "source_used": decision.route,
    }


def route_after_router(state: AgentState) -> Literal["retrieve_kb", "direct_answer"]:
    if state["source_used"] == "kb":
        return "retrieve_kb"
    return "direct_answer"

# Retrieve from Private KB



In [17]:
def retrieve_kb(state: AgentState):
    query=state["current_query"]
    docs=retriever.invoke(query)

    print(f"[KB Retriever] Query: {query} ")
    print(f"[KB Retriever] Retrieved: {len(docs)} chunks ")
    return {"kb_docs": docs}


# Node 3 - Grade Private KB Evidence


In [18]:
kb_grader_llm = llm.with_structured_output(EvidenceGrade, method="json_mode")

def grade_kb_evidence(state: AgentState):
    question = state["question"]

    context = "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n{doc.page_content}"
        for doc in state["kb_docs"]
    )

    grade = kb_grader_llm.invoke(f'''
You are an evidence grader.

Question:
{question}

Private KB evidence:
{context}

Can this private KB evidence answer the question?
Return "good" if it can answer.
Return "weak" if it cannot answer or is incomplete.

Return your response as valid JSON.
Example:
{{"grade": "good"}}
''')

    print("[KB Grader]", grade.grade)

    return {"kb_grade": grade.grade}


def decide_after_kb_grade(state: AgentState) -> Literal["generate_from_kb", "search_web"]:
    if state["kb_grade"] == "good":
        return "generate_from_kb"
    return "search_web"

# 14. Node 4 — Tavily Web Search Fallback

In [19]:
def search_web(state: AgentState):
    query=state["current_query"]

    print(f"[Tavily Search] Query: {query}")

    search_tool = globals().get("web_search") or globals().get("websearch")
    result = search_tool.invoke({"query": query})

    # Tavily can return dict/list structures depending on package version.websearch
    # Convert it into readable text for rading and generation

    if isinstance(result,dict):
        answer=result.get("answer", "")
        results = result.get("results",[])
        lines=[]
        if answer:
            lines.append(f"Tavily answer: {answer}")


        for i in results:
            title=i.get("title", "")
            url=i.get("url", "")
            content=i.get("content", "")
            lines.append(f"Title: {title}\n URL: {url}\n Content: {content}")

        web_text= "\n\n".join(lines) if lines else str(result)
    else:
        web_text=str(result)

    print("[Tavily Search] Result characters:", len(web_text))

    return {
        "web_results": web_text,
        "source_used": "web"
    }



# 15. Node 5 — Grade Web Evidence

In [20]:
web_grader_llm = llm.with_structured_output(EvidenceGrade, method="json_mode")

def grade_web_evidence(state: AgentState):
    question = state["question"]
    web_results = state["web_results"]

    grade = web_grader_llm.invoke(f'''
You are an evidence grader.

Question:
{question}

Web search evidence:
{web_results}

Can this web evidence answer the question?
Return "good" if it can answer.
Return "weak" if it cannot answer or is incomplete.

Return your response as valid JSON.
Example:
{{"grade": "good"}}
''')

    print("[Web Grader]", grade.grade)

    return {"web_grade": grade.grade}


MAX_RETRIES = 1

def decide_after_web_grade(state: AgentState) -> Literal["generate_from_web", "rewrite_query", "answer_insufficient"]:
    if state["web_grade"] == "good":
        return "generate_from_web"

    if state["retry_count"] < MAX_RETRIES:
        return "rewrite_query"

    return "answer_insufficient"

In [ ]:
# Remaining Nodes

def rewrite_query(state: AgentState):
    question = state["question"]
    current_query = state["current_query"]
    print(f"[Query Rewriter] Rewriting query: '{current_query}'")

    prompt = f'''You are an expert query optimizer for search engines.
Original question: {question}
Current search query: {current_query}

Rewrite this into an improved search query that is likely to find high-quality factual information.
Return ONLY the improved query text, no explanations or quotes.'''

    better_query = llm.invoke(prompt).content.strip()

    print(f"[Query Rewriter] New query: '{better_query}'")
    return {
        "current_query": better_query,
        "retry_count": state["retry_count"] + 1
    }


def generate_from_kb(state: AgentState):
    question = state["question"]
    context = "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n{doc.page_content}"
        for doc in state["kb_docs"]
    )
    print("[Generator] Generating answer using Private KB docs")

    prompt = f'''You are a helpful assistant. Answer the user question using ONLY the provided private KB context.

Question:
{question}

Context:
{context}

Answer clearly, accurately, and cite sources where appropriate.'''

    answer = llm.invoke(prompt).content

    return {"answer": answer, "source_used": "kb"}


def generate_from_web(state: AgentState):
    question = state["question"]
    web_results = state["web_results"]
    print("[Generator] Generating answer using Web Search results")

    prompt = f'''You are a helpful assistant. Answer the user question using the web search evidence below.

Question:
{question}

Web Search Evidence:
{web_results}

Provide a concise, factual, and direct answer.'''

    answer = llm.invoke(prompt).content

    return {"answer": answer, "source_used": "web"}


def direct_answer(state: AgentState):
    question = state["question"]
    print("[Generator] Generating direct response (no retrieval needed)")
    answer = llm.invoke(question).content
    return {"answer": answer, "source_used": "direct"}


def answer_insufficient(state: AgentState):
    print("[Generator] Evidence insufficient fallback")
    fallback_message = "I was unable to find sufficient or reliable information in both the private knowledge base and external web search to answer your question accurately."
    return {"answer": fallback_message, "source_used": "insufficient"}


In [ ]:
from langgraph.graph import StateGraph, START, END

# Assemble the Agentic RAG Graph
workflow = StateGraph(AgentState)

# 1. Register Nodes
workflow.add_node("route_question", route_question)
workflow.add_node("retrieve_kb", retrieve_kb)
workflow.add_node("grade_kb_evidence", grade_kb_evidence)
workflow.add_node("search_web", search_web)
workflow.add_node("grade_web_evidence", grade_web_evidence)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("generate_from_kb", generate_from_kb)
workflow.add_node("generate_from_web", generate_from_web)
workflow.add_node("direct_answer", direct_answer)
workflow.add_node("answer_insufficient", answer_insufficient)

# 2. Add Entry Point and Edges
workflow.add_edge(START, "route_question")

# Router conditional edge
workflow.add_conditional_edges(
    "route_question",
    route_after_router,
    {
        "retrieve_kb": "retrieve_kb",
        "direct_answer": "direct_answer",
    }
)

# KB branch
workflow.add_edge("retrieve_kb", "grade_kb_evidence")
workflow.add_conditional_edges(
    "grade_kb_evidence",
    decide_after_kb_grade,
    {
        "generate_from_kb": "generate_from_kb",
        "search_web": "search_web",
    }
)

# Web Search Fallback branch
workflow.add_edge("search_web", "grade_web_evidence")
workflow.add_conditional_edges(
    "grade_web_evidence",
    decide_after_web_grade,
    {
        "generate_from_web": "generate_from_web",
        "rewrite_query": "rewrite_query",
        "answer_insufficient": "answer_insufficient",
    }
)

# Query rewriting loop back to web search
workflow.add_edge("rewrite_query", "search_web")

# Terminal nodes to END
workflow.add_edge("generate_from_kb", END)
workflow.add_edge("generate_from_web", END)
workflow.add_edge("direct_answer", END)
workflow.add_edge("answer_insufficient", END)

# Compile the workflow
app = workflow.compile()
print("Agentic RAG workflow compiled successfully!")


In [ ]:
def ask_agent(question: str):
    print(f"\n{'='*60}")
    print(f"USER QUESTION: {question}")
    print(f"{'='*60}")

    initial_state = {
        "question": question,
        "current_query": question,
        "kb_docs": [],
        "web_results": "",
        "kb_grade": "",
        "web_grade": "",
        "answer": "",
        "source_used": "",
        "retry_count": 0,
    }

    result = app.invoke(initial_state)

    print(f"\n{'='*60}")
    print(f"SOURCE USED: {result.get('source_used')}")
    print(f"ANSWER:\n{result.get('answer')}")
    print(f"{'='*60}\n")

    return result


In [ ]:
result = ask_agent(
    "In Agentic RAG, what happens when retrieved documents are not relevant?"
)
